# Tutorial 1: Rooms and Acoustics

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/audio/1.rooms_and_acoustics.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

The [quickstart](0.quickstart.ipynb) used random rooms (`reverb="small"`, `"medium"` or `"large"`). In this tutorial, you choose **the room your dialogue is recorded in** with `Room`, and see how each choice changes the audio.

In this tutorial you will learn how to:
- simulate a room of your choice and read back what was simulated,
- set how reverberant it is, from a target reverberation time (RT60) or from the materials of its surfaces,
- place the microphone and the speakers, and use directional microphones,
- put each speaker in their own room, e.g. for both ends of a phone call,
- use measured impulse responses,
- record in real rooms measured in BUT ReverbDB, with their own background noise,
- vary the rooms across a whole dataset.

> `Room` only needs pyroomacoustics, which is installed with `sdialog[audio]`. Like the quickstart, this tutorial uses the [Dia2](https://github.com/nari-labs/dia2) TTS engine, which needs Python >= 3.10 and works best on a GPU (on Colab, select a GPU runtime).

## Environment Setup

In [ ]:
# Setup the environment depending on whether we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")
    # Installing sdialog
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %cd ..
else:
    print("Running in Jupyter Notebook")

### Load the example dialogue

We use the doctor-patient dialogue of the quickstart. To keep each example quick, we only render its first four turns.

To hear only the effect of the room, every example uses the same `seed`: the voices and the timing are then the same each time, and only the acoustics change.

In [ ]:
from sdialog import Dialog
from sdialog.audio import Room

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

dialog = Dialog.from_file(path_dialog)
excerpt = dialog[:4]  # the first four turns
excerpt.print()


def render(**kwargs):
    """Renders the excerpt with the same voices and timing every time (same seed)."""
    return excerpt.to_audio(seed=0, **kwargs)

## 1. Random rooms or your own room

With `reverb="medium"`, the room size, its reverberation time and the positions are drawn at random. With a `Room`, you choose them. Anything you don't give is still drawn at random from the seed, here the positions of the speakers:

In [ ]:
random_room = render(reverb="medium")
print(random_room.config["room"])
random_room.display()

In [ ]:
room = Room(dimensions=(5, 4, 2.8),  # width (x), length (y) and height (z), in meters
            rt60=0.5)                 # reverberation time, in seconds
my_room = render(reverb=room)
print(my_room.config["room"])
my_room.display()

`audio.config["room"]` describes the simulated room, including the values drawn at random: the microphone and speaker positions (in meters), the absorption, and the microphone directivity. It is also saved in the annotations (`audio.save()`), so every recording of a dataset documents its own room.

To see where everyone is, let's define a small helper that draws the room from above:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_room(room_info, title=None):
    """Draws a simulated room from above (from `audio.config["room"]`)."""
    width, length, height = room_info["dimensions"]
    fig, ax = plt.subplots(figsize=(4, 4 * length / width))
    ax.add_patch(plt.Rectangle((0, 0), width, length, fill=False, linewidth=2))
    mic = room_info["microphone"]
    ax.plot(mic[0], mic[1], "k^", markersize=12, label=f"microphone ({mic[2]} m high)")
    direction = room_info["mic_direction"]
    if direction is not None and direction[1] > 10:  # directional microphone, not pointing straight up
        azimuth = np.radians(direction[0])
        ax.arrow(mic[0], mic[1], 0.6 * np.cos(azimuth), 0.6 * np.sin(azimuth), head_width=0.12, color="k")
    for name, (x, y, z) in zip(room_info["speakers"], room_info["sources"]):
        ax.plot(x, y, "o", markersize=12, label=f"{name} ({z} m high)")
    ax.set_xlim(-0.3, width + 0.3)
    ax.set_ylim(-0.3, length + 0.3)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title or f"{width} x {length} x {height} m")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.show()


plot_room(my_room.config["room"])

## 2. How reverberant is the room?

The reverberation depends on how much sound the surfaces absorb. You can set it in two ways:

- `rt60`: the target reverberation time in seconds (the time the sound takes to decay by 60 dB); the absorption is computed for you.
- `materials`: the surfaces' materials, from the [pyroomacoustics database](https://github.com/LCAV/pyroomacoustics/blob/master/pyroomacoustics/data/materials.json) (e.g. `"brickwork"`, `"plasterboard"`, `"carpet_thin"`), for all surfaces or per surface (`"walls"`, `"floor"`, `"ceiling"`, or a single wall: `"east"`, `"west"`, `"north"`, `"south"`).

Let's compare four versions of the same room, with the speakers at fixed positions. `room.compute_rirs()` returns the impulse response from each speaker to the microphone, without generating any speech, so we can look at it directly:

In [ ]:
dimensions = (5, 4, 2.8)
speakers = {"Marie": (1.5, 2, 1.2), "Dr. Martin": (3.5, 2, 1.2)}

rooms = {
    "dry (RT60 0.3 s)": Room(dimensions=dimensions, speakers=speakers, rt60=0.3),
    "reverberant (RT60 1.0 s)": Room(dimensions=dimensions, speakers=speakers, rt60=1.0),
    "bare concrete": Room(dimensions=dimensions, speakers=speakers, materials="rough_concrete"),
    "office": Room(dimensions=dimensions, speakers=speakers,
                   materials={"walls": "plasterboard", "floor": "carpet_thin", "ceiling": "ceiling_fibre_absorber"}),
}

fig, axes = plt.subplots(len(rooms), 1, figsize=(8, 1.6 * len(rooms)), sharex=True)
for ax, (name, room) in zip(axes, rooms.items()):
    rirs, info = room.compute_rirs(["Marie", "Dr. Martin"], sample_rate=16000)
    ax.plot(np.arange(len(rirs[0])) / 16000, rirs[0], linewidth=0.8)
    ax.set_title(f"{name}: Marie to the microphone", fontsize=10)
axes[-1].set_xlim(0, 1.0)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

The longer the tail, the more reverberant the room. Now let's listen to each of them:

In [ ]:
for name, room in rooms.items():
    print(name)
    render(reverb=room).display()

## 3. Microphone and speakers

The microphone is on the ceiling by default. `mic` can also be:

- `(x, y, z)` coordinates,
- `"wall"`: on the middle of a wall, at standing height,
- `"between_speakers"`: between the speakers, a bit lower than their mouths, like a microphone on a table,
- a speaker's name (or index): a lapel microphone on that speaker's chest.

Let's record the same conversation from three places in the same room:

In [ ]:
mics = {
    "ceiling": Room(dimensions=dimensions, speakers=speakers, rt60=0.6),
    "table": Room(dimensions=dimensions, speakers=speakers, rt60=0.6, mic="between_speakers"),
    "lapel on Dr. Martin": Room(dimensions=dimensions, speakers=speakers, rt60=0.6, mic="Dr. Martin"),
}

for name, room in mics.items():
    audio = render(reverb=room)
    plot_room(audio.config["room"], title=f"Microphone: {name}")
    audio.display()

With the lapel microphone, Dr. Martin is close and clear while Marie sounds distant and reverberant.

### Directional microphones

Microphones are omnidirectional by default: they pick up sound equally from every direction. With `mic_pattern` (`"cardioid"`, `"subcardioid"`, `"hypercardioid"` or `"figure8"`), the microphone favors the direction it points to: `mic_direction` takes a speaker's name, `"speakers"` (the midpoint between them, the default), or an `(azimuth, colatitude)` direction in degrees.

Here, a cardioid table microphone points at Dr. Martin, so Marie, behind it, is picked up mostly through the room's reflections:

In [ ]:
cardioid = Room(dimensions=dimensions, speakers=speakers, rt60=0.6,
                mic="between_speakers", mic_pattern="cardioid", mic_direction="Dr. Martin")

rirs, info = cardioid.compute_rirs(["Marie", "Dr. Martin"], sample_rate=16000)
print("Direct sound level, Marie vs. Dr. Martin:", [round(float(np.max(np.abs(rir))), 2) for rir in rirs])

audio = render(reverb=cardioid)
plot_room(audio.config["room"], title="Cardioid microphone pointing at Dr. Martin")
audio.display()

## 4. Both ends of a phone call

Like the other scene parameters, `reverb` can be given per speaker. Each speaker is then in their own room, which, together with a telephone channel, simulates both ends of a call. Here, Marie calls from her living room with her phone in her pocket, through a mobile network (AMR), while Dr. Martin answers with a lapel microphone in his small, treated office. In stereo, each speaker is on their own channel:

In [ ]:
call = render(
    reverb={"Marie": Room(dimensions=(6, 4.5, 2.6), materials={"walls": "brickwork", "floor": "carpet_thin"},
                          mic="Marie"),
            "Dr. Martin": Room(dimensions=(3, 3, 2.6), rt60=0.25, mic="Dr. Martin")},
    noise={"Marie": 20},       # background noise on Marie's side only (SNR in dB)
    channel={"Marie": "amr"},  # Marie's voice goes through a mobile phone codec
    stereo=True,
)
for name, room_info in zip(call.speakers, call.config["room"]):
    print(name, room_info["dimensions"], room_info["materials"] or f"RT60 {room_info['rt60']} s")
call.display()

## 5. Measured impulse responses

Simulated rooms are shoeboxes. For real rooms, `reverb` also accepts measured impulse responses: an audio file, or a folder of them (each speaker then gets one at random). Public collections include the [Room Impulse Response and Noise Database](https://www.openslr.org/28/) (OpenSLR 28), and BUT ReverbDB, which SDialog reads directly (next section).

As an example, we save a simulated impulse response of a large hall to a folder, and use it as if it were a measured one:

In [ ]:
import soundfile as sf

os.makedirs("output/rooms/irs", exist_ok=True)
rirs, _ = Room(dimensions=(15, 10, 6), rt60=1.5).compute_rirs(["a"], sample_rate=16000, rng=np.random.default_rng(0))
sf.write("output/rooms/irs/hall.wav", rirs[0], 16000)

measured = render(reverb="output/rooms/irs/")
print(measured.config["room"])
measured.display()

## 6. Real rooms: BUT ReverbDB

[BUT ReverbDB](https://speech.fit.vut.cz/software/but-speech-fit-reverb-database) has impulse responses measured in 9 real rooms (offices, meeting and lecture rooms, a staircase, hotel rooms), from several loudspeaker positions to 31 microphones, plus each room's own background noise recorded at every microphone. Real rooms have furniture, irregular shapes and real microphones, which a simulated shoebox does not.

SDialog does not include the database (CC BY 4.0). Download `BUT_ReverbDB_rel_19_06_RIR-Only.tgz` (8.7 GB) and extract it into a folder of its own, as the archive has no top folder. Uncomment the download lines below, or set `BUT_PATH` to where you already have it. Without it, this section is skipped.


In [ ]:
from sdialog.audio import ButReverbDB

BUT_PATH = os.environ.get("BUT_REVERBDB", "BUT_ReverbDB")  # the folder where the archive was extracted

# !wget http://merlin.fit.vutbr.cz/ReverbDB/BUT_ReverbDB_rel_19_06_RIR-Only.tgz
# !mkdir -p BUT_ReverbDB && tar xzf BUT_ReverbDB_rel_19_06_RIR-Only.tgz -C BUT_ReverbDB

if os.path.isdir(BUT_PATH):
    but = ButReverbDB(BUT_PATH)
    print(but)
    print(but.rooms)
else:
    but = None
    print(f"BUT ReverbDB not found in '{BUT_PATH}': skipping this section.")


With `reverb=but`, each dialogue is recorded in a random room by a random microphone, and each speaker gets a different loudspeaker position in that room. With `noise="room"`, the background noise is the one recorded by that same microphone, so the reverberation and the noise come from the same place, as in a real recording:


In [ ]:
if but is not None:
    real_room = render(reverb=but, noise={"type": "room", "snr": (10, 20)})
    room_info = real_room.config["room"]
    print("Room:", room_info["room"], "| microphone:", room_info["microphone"])
    print("Loudspeaker positions:", room_info["speakers"])
    print("Noise:", real_room.config["noise"])
    real_room.display()


`rooms=` restricts the rooms to use. Let's compare a small office with a large lecture room:


In [ ]:
if but is not None:
    for name in ["VUT_FIT_L207", "VUT_FIT_D105"]:  # office (98 m³) and lecture room (~2000 m³)
        print(name)
        render(reverb=ButReverbDB(BUT_PATH, rooms=[name])).display()


Like the other scene parameters, it can also be given per speaker. As in section 4, here Marie calls from a real room, with its noise, through a mobile network, while Dr. Martin's side stays clean:


In [ ]:
if but is not None:
    call = render(reverb={"Marie": but}, noise={"Marie": "room"}, channel={"Marie": "amr"}, stereo=True)
    print(call.config["room"][call.speakers.index("Marie")]["room"])
    call.display()


> `noise="room"` needs the reverb of the same speakers to be a `ButReverbDB` (a shared noise needs a shared reverb, a speaker's noise needs that speaker's reverb); otherwise it raises an error instead of leaving the audio without noise.
>
> When publishing results obtained with BUT ReverbDB, please cite: I. Szöke, M. Skácel, L. Mošner, J. Paliesek and J. Černocký, "Building and Evaluation of a Real Room Impulse Response Dataset", *IEEE Journal of Selected Topics in Signal Processing*, 13(4), 863-876, 2019.


## 7. Varying the rooms across a dataset

A list of values means "pick one at random", so a list of rooms gives each dialogue one of them, and a `Room` without speaker positions places the speakers differently in each dialogue. With `dialogs_to_audio`, each dialogue gets its own seed, so the whole dataset is reproducible:

In [ ]:
import json
from sdialog.audio import dialogs_to_audio

rooms = [
    Room(dimensions=(4, 3.5, 2.6), rt60=0.4, mic="between_speakers"),               # small consultation room
    Room(dimensions=(8, 6, 3), materials="plasterboard", mic="wall"),                # meeting room
    Room(dimensions=(12, 9, 4), rt60=1.0, mic_pattern="cardioid"),                   # large hall, cardioid microphone
]

dialogs = [excerpt.clone() for _ in range(3)]  # e.g. a list of generated dialogues (each with its own id)
rows = dialogs_to_audio(dialogs, "output/rooms/batch", reverb=rooms, noise=(20, 30), seed=0)

for row in rows:
    with open(row["json"]) as f:
        room_info = json.load(f)["config"]["room"]
    print(row["audio"], room_info["dimensions"], "microphone at", room_info["microphone"])

## Learn more

- [Custom Rooms](https://sdialog.readthedocs.io/en/latest/sdialog/index.html#custom-rooms) in the documentation, and the [`Room` API reference](https://sdialog.readthedocs.io/en/latest/api/sdialog.html#module-sdialog.audio.room).
- [Measured Rooms (BUT ReverbDB)](https://sdialog.readthedocs.io/en/latest/sdialog/index.html#measured-rooms-but-reverbdb) in the documentation, and the [`ButReverbDB` API reference](https://sdialog.readthedocs.io/en/latest/api/sdialog.html#module-sdialog.audio.but).
- Coming from the legacy audio pipeline (`AudioPipeline`, room generators)? See [how each of its features maps to `dialog.to_audio()`](https://sdialog.readthedocs.io/en/latest/sdialog/index.html#coming-from-the-legacy-audio-pipeline).